# **Hyperparameter Tuning for the GA Meal Planner**

In [ ]:
import json
import sys
from datetime import datetime, timedelta
from random import Random

from optuna import create_study
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
)

sys.path.append("..")

from engines import GAMealPlanner, load_all_ingredients, make_preferences
from models import MealPlanningEnvironment, Pantry, PantryIngredient

The default user preferences are set for tuning the GA Meal Planner.

In [2]:
preferences = make_preferences()

In [3]:
all_ingredients = load_all_ingredients("../recipe_extraction/supplemented_structured_ingredients.json")

In [4]:
SEED = 1

PANTRY_SIZE = 40

QUANTITY_MIN = 200
QUANTITY_MAX = 2000

EXPIRY_DAYS_MIN = 1
EXPIRY_DAYS_MAX = 14

In [5]:
rng = Random(SEED)

CURRENT_DATE = datetime.now()

sampled_ingredients = rng.sample(all_ingredients, PANTRY_SIZE)

pantry = Pantry()

for ingredient in sampled_ingredients:
    quantity = rng.randint(QUANTITY_MIN, QUANTITY_MAX)
    expiry_days = rng.randint(EXPIRY_DAYS_MIN, EXPIRY_DAYS_MAX)

    pantry_ingredient = PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=CURRENT_DATE + timedelta(days=expiry_days),
    )
    pantry.add(pantry_ingredient, quantity)

meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

pantry.print(3)

---
Quantity: 252 g
Ingredient: unsalted kosher-for-Passover pareve (non-dairy) margarine
	Estimated Expiration Date: 2026-05-11
	Nutritional Information:
		Calories: 714.0 kcal
		Carbohydrates: 0.0 g
		Sugar: None g
		Protein: 0.0 g
		Fat: 78.57 g
		Saturated Fat: 10.71 g
		Fiber: None g
		Sodium: 0.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1308 g
Ingredient: white truffles
	Estimated Expiration Date: 2026-05-01
	Nutritional Information:
		Calories: 639.0 kcal
		Carbohydrates: 38.89 g
		Sugar: 38.89 g
		Protein: 5.56 g
		Fat: 52.78 g
		Saturated Fat: 36.11 g
		Fiber: 0.0 g
		Sodium: 83.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 980 g
Ingredient: orange wheel
	Estimated Expiration Date: 2026-05-11
	Nutritional Information:
		Calories: 381.0 kcal
		Carbohydrates: 90.48 g
		Sugar: 90.48 g
		Protein: 4.76 g
		Fat: 0.0 g
		Saturated Fat: None g
		Fiber: None g
		Sodium: 381.0 mg
		Gluten Fr

In [6]:
all_ingredient_names = []
for recipe in meal_planning_environment.recipes:
    for ingredient_name in recipe.ingredients.keys():
        all_ingredient_names.append(ingredient_name)

all_ingredient_costs = {}
for ingredient_name in sorted(set(all_ingredient_names)):
    all_ingredient_costs[ingredient_name] = rng.random()

meal_planning_environment.ingredient_costs = all_ingredient_costs
meal_planning_environment._check_ingredient_costs()

In [7]:
sampler = TPESampler(seed=SEED)

In [8]:
study = create_study(direction="maximize", study_name="GA Meal Planner Hyperparameter Tuning", sampler=sampler)

[I 2026-04-30 15:18:49,517] A new study created in memory with name: GA Meal Planner Hyperparameter Tuning


In [9]:
NUM_DAYS = 7
NUM_MEALS_PER_DAY = 3
TUNING_SEED = 1

# fixed at 50 to keep tuning time reasonable. Increasing num generations is almost but guaranteed to increase performance, so this is more of a practical choice to keep tuning time manageable
TUNING_NUM_GENERATIONS = 50

In [10]:
def optimise(trial: Trial) -> float:
    """
    Optimisation objective function for the GA Meal Planner

    Tunes only the GA operator hyperparameters (selection, crossover, mutation operators and their settings)

    The generation count is intentionally fixed at a constant low budget so that all trials are comparable and Optuna cannot trivially maximise fitness by simply preferring more generations

    :param trial: an Optuna trial object
    :type trial: optuna.trial.Trial

    :return: the fitness score of the best meal plan generated by the GA Meal Planner with the given hyperparameters
    :rtype: float
    """

    population_size = trial.suggest_int("population_size", 20, 200, step=20)
    num_parents_mating = trial.suggest_int("num_parents_mating", 2, population_size, step=2)
    parent_selection_type = trial.suggest_categorical(
        "parent_selection_type", ["sss", "rws", "sus", "random", "tournament"]
    )
    k_tournament = trial.suggest_int("K_tournament", 2, 10)
    keep_elitism = trial.suggest_int("keep_elitism", 0, 1)
    crossover_type = trial.suggest_categorical("crossover_type", ["single_point", "two_points", "uniform", "scattered"])
    crossover_probability = trial.suggest_float("crossover_probability", 0.5, 1.0, step=0.1)
    mutation_type = trial.suggest_categorical("mutation_type", ["random", "swap", "inversion", "scramble"])
    mutation_probability = trial.suggest_float("mutation_probability", 0.01, 0.5, step=0.01)

    ga_meal_planner = GAMealPlanner(meal_planning_environment)

    _, best_fitness = ga_meal_planner.generate_meal_plan(
        num_days=NUM_DAYS,
        meals_per_day=NUM_MEALS_PER_DAY,
        num_generations=TUNING_NUM_GENERATIONS,
        num_parents_mating=num_parents_mating,
        population_size=population_size,
        parent_selection_type=parent_selection_type,
        K_tournament=k_tournament,
        keep_elitism=keep_elitism,
        crossover_type=crossover_type,
        crossover_probability=crossover_probability,
        mutation_type=mutation_type,
        mutation_probability=mutation_probability,
        generation_print_interval=None,
        seed=TUNING_SEED,
    )

    return best_fitness

In [12]:
NUM_TRIALS = 500

In [13]:
study.optimize(
    optimise,
    n_trials=NUM_TRIALS,
    show_progress_bar=True,
)

print(f"\nBest trial: #{study.best_trial.number}")
print(f"Best fitness score: {study.best_value:.4f}")

  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-04-30 15:19:00,422] Trial 0 finished with value: -1.5961681377646486 and parameters: {'population_size': 100, 'num_parents_mating': 74, 'parent_selection_type': 'rws', 'K_tournament': 5, 'keep_elitism': 0, 'crossover_type': 'uniform', 'crossover_probability': 1.0, 'mutation_type': 'swap', 'mutation_probability': 0.08}. Best is trial 0 with value: -1.5961681377646486.
[I 2026-04-30 15:19:01,173] Trial 1 finished with value: -0.5322155486713054 and parameters: {'population_size': 40, 'num_parents_mating': 34, 'parent_selection_type': 'sss', 'K_tournament': 2, 'keep_elitism': 0, 'crossover_type': 'two_points', 'crossover_probability': 1.0, 'mutation_type': 'swap', 'mutation_probability': 0.42000000000000004}. Best is trial 1 with value: -0.5322155486713054.
[I 2026-04-30 15:19:01,607] Trial 2 finished with value: -0.4017017175350027 and parameters: {'population_size': 20, 'num_parents_mating': 16, 'parent_selection_type': 'sss', 'K_tournament': 6, 'keep_elitism': 1, 'crossover_typ

In [14]:
best_params = study.best_params

print("Best hyperparameters:")
for key, value in best_params.items():
    print(f"\t- {key}: {value}")

Best hyperparameters:
	- population_size: 180
	- num_parents_mating: 48
	- parent_selection_type: sss
	- K_tournament: 3
	- keep_elitism: 0
	- crossover_type: single_point
	- crossover_probability: 1.0
	- mutation_type: inversion
	- mutation_probability: 0.09


In [15]:
FINAL_NUM_GENERATIONS = 500

In [16]:
ga_final = GAMealPlanner(meal_planning_environment)

_, final_fitness = ga_final.generate_meal_plan(
    num_days=NUM_DAYS,
    meals_per_day=NUM_MEALS_PER_DAY,
    num_generations=FINAL_NUM_GENERATIONS,
    num_parents_mating=best_params["num_parents_mating"],
    population_size=best_params["population_size"],
    parent_selection_type=best_params["parent_selection_type"],
    K_tournament=best_params["K_tournament"],
    keep_elitism=best_params["keep_elitism"],
    crossover_type=best_params["crossover_type"],
    crossover_probability=best_params["crossover_probability"],
    mutation_type=best_params["mutation_type"],
    mutation_probability=best_params["mutation_probability"],
    generation_print_interval=50,
    seed=TUNING_SEED,
)

print(f"\nFinal fitness score ({FINAL_NUM_GENERATIONS} generations): {final_fitness:.4f}")
print(f"Tuning fitness score ({TUNING_NUM_GENERATIONS} generations): {study.best_value:.4f}")

Generation 50, Best Fitness: -0.24
Generation 100, Best Fitness: -0.24
Generation 150, Best Fitness: -0.24
Generation 200, Best Fitness: -0.24
Generation 250, Best Fitness: -0.24
Generation 300, Best Fitness: -0.24
Generation 350, Best Fitness: -0.24
Generation 400, Best Fitness: -0.24
Generation 450, Best Fitness: -0.24
Generation 500, Best Fitness: -0.24

Final fitness score (500 generations): -0.2353
Tuning fitness score (50 generations): -0.2353


In [21]:
with open("best_ga_meal_planner_hyperparameters.json", "w") as file:
    json.dump(best_params, file, indent=4)

In [22]:
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values("value", ascending=False).reset_index(drop=True)
trials_df

,number,value,datetime_start,datetime_complete,duration,params_K_tournament,params_crossover_probability,params_crossover_type,params_keep_elitism,params_mutation_probability,params_mutation_type,params_num_parents_mating,params_parent_selection_type,params_population_size,state
0,459,-0.235349,2026-04-30 15:58:36.488506,2026-04-30 15:58:39.843406,0 days 00:00:03.354900,7,1.0,single_point,0,0.14,inversion,48,sss,180,COMPLETE
1,411,-0.235349,2026-04-30 15:54:44.701055,2026-04-30 15:54:48.047908,0 days 00:00:03.346853,5,1.0,single_point,0,0.15,inversion,48,sss,180,COMPLETE
2,350,-0.235349,2026-04-30 15:49:47.075974,2026-04-30 15:49:50.426768,0 days 00:00:03.350794,4,1.0,single_point,0,0.10,inversion,48,sss,180,COMPLETE
3,213,-0.235349,2026-04-30 15:39:35.914452,2026-04-30 15:39:39.314460,0 days 00:00:03.400008,4,1.0,single_point,0,0.14,inversion,48,sss,180,COMPLETE
4,231,-0.235349,2026-04-30 15:40:36.778451,2026-04-30 15:40:40.162769,0 days 00:00:03.384318,4,1.0,single_point,0,0.13,inversion,48,sss,180,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,275,-2.048656,2026-04-30 15:44:27.118505,2026-04-30 15:44:30.911861,0 days 00:00:03.793356,4,1.0,single_point,0,0.12,inversion,42,rws,180,COMPLETE
496,458,-2.048656,2026-04-30 15:58:32.933318,2026-04-30 15:58:36.486987,0 days 00:00:03.553669,4,1.0,single_point,0,0.11,inversion,42,rws,180,COMPLETE
497,322,-2.067533,2026-04-30 15:48:05.377993,2026-04-30 15:48:08.913576,0 days 00:00:03.535583,3,1.0,single_point,0,0.14,inversion,44,sus,180,COMPLETE
498,430,-2.067533,2026-04-30 15:56:26.000875,2026-04-30 15:56:29.568327,0 days 00:00:03.567452,2,1.0,single_point,0,0.10,inversion,44,sus,180,COMPLETE


In [23]:
plot_optimization_history(study).show()

In [24]:
plot_param_importances(study).show()